<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 10: Google QuickDraw Çizim Tanıma

**YAPAY ZEKA MÜHENDİSLİĞİ** · Modül 10 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta10/hafta10_quickdraw.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta10/hafta10_quickdraw.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>&nbsp;
<a href="https://raw.githubusercontent.com/DrMuratAltun/VB-YZ-90/main/web/public/sunumlar/hafta10_derin_ogrenme.pdf"><img src="https://img.shields.io/badge/PDF%20Sunum-EC1C24?style=flat&logo=adobeacrobatreader&logoColor=white" alt="PDF Sunum"/></a>&nbsp;
<a href="https://drmurataltun.github.io/VB-YZ-90/hafta/10/"><img src="https://img.shields.io/badge/Web%20Sitesi-2B7A78?style=flat&logo=googlechrome&logoColor=white" alt="Web Sitesi"/></a>

</div>

---

**Eğitmen:** Dr. Murat Altun · [yapayzekaokulum.com](https://yapayzekaokulum.com) · [GitHub](https://github.com/DrMuratAltun)

**Program:** ECS Veri Bilimi ve Yapay Zeka Uzmanlığı · 90 Saat · 15 Hafta
---

> **Bu defterde neler öğreneceksiniz?**
>
> - Google QuickDraw veri seti
> - CNN ile çizim tanıma
> - Data preprocessing ve augmentation

# Hafta 10 - QuickDraw Benzeri Şekil Tanıma

## Google Quick, Draw! Veri Seti Nedir?

[Quick, Draw!](https://quickdraw.withgoogle.com/) Google'ın geliştirdiği bir oyundur. Kullanıcılardan belirli nesneleri 20 saniye içinde çizmesi istenir ve bir yapay sinir ağı çizimi tanımaya çalışır.

**Veri seti özellikleri:**
- 345 farklı kategori (kedi, araba, ev, ağaç vb.)
- Milyonlarca el çizimi
- 28x28 piksel gri tonlamalı görüntüler olarak sunulabilir

Bu defterde:
1. NumPy ile sentetik şekil verileri üreteceğiz (daire, kare, üçgen, çizgi, yıldız)
2. Fashion MNIST'i alternatif veri seti olarak kullanacağız
3. ANN modeli eğiteceğiz
4. Tahminleri görselleştireceğiz

## 1. Kütüphaneler

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `matplotlib` | Grafik ve görselleştirme |
| `numpy` | Sayısal hesaplamalar ve dizi işlemleri |
| `sklearn` | Makine öğrenmesi algoritmaları ve araçları |
| `tensorflow` | Derin öğrenme modelleri oluşturma ve eğitme |
| `warnings` | Uyarı mesajlarını yönetme |


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow sürümü: {tf.__version__}")

## Bölüm A: Sentetik Şekil Verisi Üretimi

NumPy kullanarak 28x28 boyutunda basit geometrik şekiller oluşturacağız:
- **0:** Daire
- **1:** Kare
- **2:** Üçgen
- **3:** Çizgi
- **4:** Yıldız

In [ ]:
def daire_ciz(boyut=28):
    """Rastgele boyut ve konumda daire çizer."""
    img = np.zeros((boyut, boyut), dtype=np.float32)
    cx = np.random.randint(8, 20)
    cy = np.random.randint(8, 20)
    r = np.random.randint(4, 9)
    for i in range(boyut):
        for j in range(boyut):
            dist = np.sqrt((i - cy)**2 + (j - cx)**2)
            if abs(dist - r) < 1.2:
                img[i, j] = 1.0
    return img

def kare_ciz(boyut=28):
    """Rastgele boyut ve konumda kare çizer."""
    img = np.zeros((boyut, boyut), dtype=np.float32)
    x1 = np.random.randint(3, 10)
    y1 = np.random.randint(3, 10)
    kenar = np.random.randint(8, 16)
    x2 = min(x1 + kenar, boyut - 2)
    y2 = min(y1 + kenar, boyut - 2)
    img[y1, x1:x2] = 1.0
    img[y2, x1:x2] = 1.0
    img[y1:y2, x1] = 1.0
    img[y1:y2, x2] = 1.0
    return img

def ucgen_ciz(boyut=28):
    """Rastgele boyutta üçgen çizer."""
    img = np.zeros((boyut, boyut), dtype=np.float32)
    taban_y = np.random.randint(18, 24)
    tepe_y = np.random.randint(3, 10)
    sol_x = np.random.randint(3, 8)
    sag_x = np.random.randint(20, 25)
    tepe_x = (sol_x + sag_x) // 2
    
    # Taban çizgisi
    img[taban_y, sol_x:sag_x] = 1.0
    
    # Sol kenar
    yukseklik = taban_y - tepe_y
    if yukseklik > 0:
        for t in np.linspace(0, 1, 50):
            y = int(taban_y - t * yukseklik)
            x = int(sol_x + t * (tepe_x - sol_x))
            if 0 <= y < boyut and 0 <= x < boyut:
                img[y, x] = 1.0
        # Sağ kenar
        for t in np.linspace(0, 1, 50):
            y = int(taban_y - t * yukseklik)
            x = int(sag_x - t * (sag_x - tepe_x))
            if 0 <= y < boyut and 0 <= x < boyut:
                img[y, x] = 1.0
    return img

def cizgi_ciz(boyut=28):
    """Rastgele açıda çizgi çizer."""
    img = np.zeros((boyut, boyut), dtype=np.float32)
    x1 = np.random.randint(2, 10)
    y1 = np.random.randint(2, 26)
    x2 = np.random.randint(18, 26)
    y2 = np.random.randint(2, 26)
    for t in np.linspace(0, 1, 80):
        x = int(x1 + t * (x2 - x1))
        y = int(y1 + t * (y2 - y1))
        if 0 <= y < boyut and 0 <= x < boyut:
            img[y, x] = 1.0
            if y + 1 < boyut:
                img[y + 1, x] = 0.5
    return img

def yildiz_ciz(boyut=28):
    """Basit 5 köşeli yıldız çizer."""
    img = np.zeros((boyut, boyut), dtype=np.float32)
    cx, cy = 14, 14
    r_dis = np.random.randint(8, 11)
    r_ic = r_dis // 2
    
    noktalar = []
    for i in range(10):
        aci = np.pi / 2 + i * np.pi / 5
        r = r_dis if i % 2 == 0 else r_ic
        x = cx + int(r * np.cos(aci))
        y = cy - int(r * np.sin(aci))
        noktalar.append((x, y))
    
    for i in range(len(noktalar)):
        x1, y1 = noktalar[i]
        x2, y2 = noktalar[(i + 1) % len(noktalar)]
        for t in np.linspace(0, 1, 40):
            x = int(x1 + t * (x2 - x1))
            y = int(y1 + t * (y2 - y1))
            if 0 <= y < boyut and 0 <= x < boyut:
                img[y, x] = 1.0
    return img

print("Şekil çizme fonksiyonları hazır!")

### Veri seti oluştur

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Veri seti oluştur
sekil_fonksiyonlari = [daire_ciz, kare_ciz, ucgen_ciz, cizgi_ciz, yildiz_ciz]
sekil_isimleri = ['Daire', 'Kare', 'Üçgen', 'Çizgi', 'Yıldız']

n_per_class = 1000  # Her sınıf için örnek sayısı
X_data = []
y_data = []

for label, fonksiyon in enumerate(sekil_fonksiyonlari):
    for _ in range(n_per_class):
        img = fonksiyon()
        X_data.append(img)
        y_data.append(label)

X_data = np.array(X_data)
y_data = np.array(y_data)

# Karıştır
indices = np.random.permutation(len(X_data))
X_data = X_data[indices]
y_data = y_data[indices]

# Eğitim/test bölme
split = int(0.8 * len(X_data))
X_train_s, X_test_s = X_data[:split], X_data[split:]
y_train_s, y_test_s = y_data[:split], y_data[split:]

print(f"Toplam veri: {len(X_data)}")
print(f"Eğitim: {len(X_train_s)}, Test: {len(X_test_s)}")
print(f"Sınıflar: {sekil_isimleri}")

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Üretilen şekillerden örnekler
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Sentetik Şekil Örnekleri', fontsize=14, fontweight='bold')

for col, (isim, fonk) in enumerate(zip(sekil_isimleri, sekil_fonksiyonlari)):
    for row in range(2):
        img = fonk()
        axes[row, col].imshow(img, cmap='gray_r')
        if row == 0:
            axes[row, col].set_title(isim, fontsize=12, fontweight='bold')
        axes[row, col].axis('off')

plt.tight_layout()
plt.show()

### Model Eğitimi

Aşağıdaki kodda modeli eğitim verisi üzerinde eğitiyoruz (`.fit()`). Eğitim sonrası test verisi üzerinde tahmin yapıp (`.predict()`) başarı metriklerini hesaplıyoruz.

In [ ]:
# Şekil tanıma modeli
model_sekil = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(5, activation='softmax')
])

model_sekil.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_s = model_sekil.fit(
    X_train_s, y_train_s,
    epochs=15,
    validation_split=0.2,
    verbose=1
)

test_loss, test_acc = model_sekil.evaluate(X_test_s, y_test_s, verbose=0)
print(f"\nŞekil Tanıma Test Doğruluğu: %{test_acc*100:.2f}")

### Dağılım Görselleştirmesi

Aşağıdaki grafikte verinin dağılımını histogram ile inceliyoruz. Dağılımın şekli (normal, çarpık, bimodal) hangi istatistiksel yöntemlerin uygulanabileceğini belirler.

In [ ]:
# Eğitim eğrileri
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_s.history['accuracy'], label='Eğitim', linewidth=2)
ax1.plot(history_s.history['val_accuracy'], label='Doğrulama', linewidth=2)
ax1.set_title('Şekil Tanıma - Doğruluk', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Doğruluk')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history_s.history['loss'], label='Eğitim', linewidth=2)
ax2.plot(history_s.history['val_loss'], label='Doğrulama', linewidth=2)
ax2.set_title('Şekil Tanıma - Kayıp', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Kayıp')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Bölüm B: Fashion MNIST ile Sınıflandırma

Fashion MNIST, MNIST'in kıyafet versiyonudur. 10 farklı giysi kategorisi içerir:

| Etiket | Sınıf |
|--------|-------|
| 0 | Tişört/Üst |
| 1 | Pantolon |
| 2 | Kazak |
| 3 | Elbise |
| 4 | Mont |
| 5 | Sandalet |
| 6 | Gömlek |
| 7 | Spor Ayakkabı |
| 8 | Çanta |
| 9 | Bot |

In [ ]:
# Fashion MNIST'i yükle
(x_train_f, y_train_f), (x_test_f, y_test_f) = tf.keras.datasets.fashion_mnist.load_data()

sinif_isimleri = ['Tişört/Üst', 'Pantolon', 'Kazak', 'Elbise', 'Mont',
                  'Sandalet', 'Gömlek', 'Spor Ayakkabı', 'Çanta', 'Bot']

# Normalizasyon
x_train_f = x_train_f / 255.0
x_test_f = x_test_f / 255.0

print(f"Eğitim: {x_train_f.shape}, Test: {x_test_f.shape}")

# Örnekleri göster
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('Fashion MNIST Örnekleri', fontsize=14, fontweight='bold')

for i in range(10):
    ax = axes[i // 5, i % 5]
    idx = np.where(y_train_f == i)[0][0]
    ax.imshow(x_train_f[idx], cmap='gray')
    ax.set_title(sinif_isimleri[i], fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

### Model Eğitimi

Aşağıdaki kodda modeli eğitim verisi üzerinde eğitiyoruz (`.fit()`). Eğitim sonrası test verisi üzerinde tahmin yapıp (`.predict()`) başarı metriklerini hesaplıyoruz.

In [ ]:
# Fashion MNIST modeli
model_fashion = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

model_fashion.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_f = model_fashion.fit(
    x_train_f, y_train_f,
    epochs=15,
    validation_split=0.2,
    verbose=1
)

test_loss_f, test_acc_f = model_fashion.evaluate(x_test_f, y_test_f, verbose=0)
print(f"\nFashion MNIST Test Doğruluğu: %{test_acc_f*100:.2f}")

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Tahminleri göster
preds_f = model_fashion.predict(x_test_f)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle('Fashion MNIST Tahminleri', fontsize=14, fontweight='bold')

indices = np.random.choice(len(x_test_f), 12, replace=False)
for i, idx in enumerate(indices):
    ax = axes[i // 4, i % 4]
    ax.imshow(x_test_f[idx], cmap='gray')
    pred = np.argmax(preds_f[idx])
    actual = y_test_f[idx]
    conf = preds_f[idx][pred] * 100
    color = 'green' if pred == actual else 'red'
    ax.set_title(f'Tahmin: {sinif_isimleri[pred]}\nGerçek: {sinif_isimleri[actual]}\nGüven: %{conf:.1f}',
                 fontsize=8, color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()

## İnteraktif Tahmin Simülasyonu

Aşağıdaki hücrede kullanıcıdan bir şekil seçmesini isteyip, o şekli üretip modelin tahminiyle karşılaştırıyoruz.

In [ ]:
# İnteraktif tahmin: rastgele bir şekil üret ve modele sor
def interaktif_tahmin():
    """Rastgele şekil üretip modele tahmin ettirir."""
    secim = np.random.randint(0, 5)
    img = sekil_fonksiyonlari[secim]()
    
    # Modele sor
    pred = model_sekil.predict(img.reshape(1, 28, 28), verbose=0)
    tahmin = np.argmax(pred)
    guven = pred[0][tahmin] * 100
    
    # Görselleştir
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    
    ax1.imshow(img, cmap='gray_r')
    ax1.set_title(f'Üretilen Şekil: {sekil_isimleri[secim]}', fontsize=13, fontweight='bold')
    ax1.axis('off')
    
    renkler = ['#2196F3' if i != tahmin else '#4CAF50' for i in range(5)]
    ax2.barh(sekil_isimleri, pred[0] * 100, color=renkler)
    ax2.set_xlabel('Güven (%)', fontsize=11)
    ax2.set_title(f'Model Tahmini: {sekil_isimleri[tahmin]} (%{guven:.1f})', fontsize=13, fontweight='bold')
    ax2.set_xlim(0, 100)
    
    dogru = '✓ DOĞRU' if tahmin == secim else '✗ YANLIŞ'
    fig.suptitle(dogru, fontsize=16, fontweight='bold',
                 color='green' if tahmin == secim else 'red', y=1.02)
    
    plt.tight_layout()
    plt.show()

# 4 kez dene
for _ in range(4):
    interaktif_tahmin()

## Özet

Bu defterde:
- NumPy ile sentetik şekil verileri ürettik (QuickDraw benzeri)
- Şekil tanıma modeli eğittik
- Fashion MNIST ile giysi sınıflandırma yaptık
- İnteraktif tahmin simülasyonu gerçekleştirdik

**Öğrenilen kavramlar:** Veri üretimi, çok sınıflı sınıflandırma, model karşılaştırma

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://scholargent.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

&copy; 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>